# Week 1 · Day 3 — Raw Dataset Ingestion
**Owner:** Taeyun · **Project:** Multi-Cloud FinOps and Infrastructure Unit Economics Engine

This notebook reads the original synthetic FOCUS and orders CSV files into a raw SQLite staging database. It intentionally does not clean data; that begins Day 4. Run from the repository root.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / "scripts/week1_day3_ingestion.py").exists():
    raise FileNotFoundError("Run Jupyter from the repository root with scripts/week1_day3_ingestion.py present")
sys.path.insert(0, str(repo_root / "scripts"))
from week1_day3_ingestion import ingest

## Source files
Place `focus_exercise_2026-q3.csv` and `northwind_orders_2026-q3.csv` inside the repository’s `focus-exercise/` directory. These files are from the same 2026 Q3 synthetic dataset, not live billing data.

In [ ]:
billing = repo_root / "focus-exercise/focus_exercise_2026-q3.csv"
orders = repo_root / "focus-exercise/northwind_orders_2026-q3.csv"
assert billing.is_file(), f"Missing {billing}"
assert orders.is_file(), f"Missing {orders}"
result = ingest(billing, orders, repo_root / "outputs")
result

## Validation
The raw file’s column order and row counts are checked, and SHA-256 fingerprints are recorded. Every source value remains text, including empty strings. This avoids accidental data transformations before cleaning.

In [ ]:
import sqlite3
with sqlite3.connect(repo_root / "outputs/finops_day3_raw_staging.sqlite") as conn:
    for table in ("raw_focus_billing", "raw_daily_orders", "ingestion_metadata"):
        print(table, conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0])
    print("Provider breakdown:", conn.execute("SELECT ProviderName, COUNT(*) FROM raw_focus_billing GROUP BY ProviderName").fetchall())
assert result["status"] == "PASS"
assert result["billing_rows"] == 4433
assert result["daily_order_rows"] == 92

## Handoff to Days 4–7
Use `raw_focus_billing` as the immutable baseline. Clean into separate tables/files, retaining the original source columns for compatibility with Gitika’s `Fact_Spend`, `Dim_Service`, `Dim_Tenant`, and `Dim_Region`. Never impute missing tags or regions without documented rules.